The purpose of this notebook is to extract data from LabChart recordings and process it to attain spike data for a single unit, then export those data for analysis in the next notebook (main_part2).

# STEP 0: IMPORT MODULES

Run this code block once per session, to initialise and import modules.

In [ ]:
import hff_analysis

# STEP 1: READ ADICHT FILE INTO PYTHON

This block only needs to be run once for each file being read,
unless `data_segments` needs to be changed.

## Arguments
* `filename` -- name of the file to be read (extension optional).
* `repetition` -- `int` for which repetition of its test the file is.
* `data_segments` -- which recording segment(s) from the file to read.
    Define a list to read segments at those indices, or set to `None`
    to read all segments in the file.
    * The duration in seconds of each segment which is read will be
      printed to assist in identifying relevant segment(s). If
      unnecessary segments have been read in this block, they can be
      filtered out in the next step without needing to rerun this block.

In [ ]:
# User defined variables
filename = 'hff20_pos1_freqsweep'
repetition = 0
data_segments = None


### User does not need to modify below this line ###

recordings = hff_analysis.read_adicht(filename, data_segments)

## STEP 1a: LOAD FILEREADSETTINGS JSON

In [ ]:
import pprint

# User defined variables
frs_filename = 'frs_hff02-1_[0-0]_freq_v2.0.0'


### User does not need to modify below this line ###

frs = hff_analysis.load_filereadsettings(frs_filename)
filename = frs.filename
repetition = frs.repetition
data_segments = [frs.recording_segment]
recording_id = 0
epoch_timing_ms = frs.epoch_timing_ms
threshold_uV = frs.threshold_uV
spike_criteria = frs.spike_criteria
exclude_frequencies = frs.exclude_frequencies
exclude_amplitudes = frs.exclude_amplitudes
print("FileReadSettings LOADED\n")
print(f"EPOCH TIMING: {epoch_timing_ms}")
print(f"THRESHOLD: {threshold_uV}")
print("SPIKE CRITERIA:")
pprint.PrettyPrinter().pprint(spike_criteria)
print(f"EXCLUDED FREQUENCIES: {exclude_frequencies}")
print(f"EXCLUDED AMPLITUDES: {exclude_amplitudes}")
recordings = hff_analysis.read_adicht(filename, data_segments)
recording = recordings[recording_id]
(spikes, epochs) = hff_analysis.spikes_info(
    recording,
    repetition,
    epoch_timing_ms,
    threshold_uV
)
recording_segment = (data_segments[recording_id] if data_segments
                     else recording_id)
filtered_spikes = hff_analysis.filter_spikes(
    spikes,
    spike_criteria,
    exclude_frequencies,
    exclude_amplitudes
)
hff_analysis.plot_clusters(
    filtered_spikes,
    epochs,
    repetition,
    recording_segment,
    exclude_frequencies,
    exclude_amplitudes,
    False
)

## STEP 1b: UPDATE EXISTING SAVE FILES

In [ ]:
input_filenames = []
input_folder = r"outputs\archive\v1.1.0"
save_outputs = 'frs'
show_plots = False


### User does not need to modify below this line ###

hff_analysis.update_outputs(
    save_outputs,
    input_filenames,
    input_folder,
    show_plots
)

# STEP 2: DETECT SPIKES

This code block detects spikes. Run this block once to get a list of
spikes which will be filtered in Step 3. There is no need to rerun this
block after moving to Step 3, unless the initial settings were too
narrow (i.e. it is better to run this with generous spike detection
settings, then apply increasingly narrow filters in Step 3 until a unit
of interest is isolated).

## Arguments
* `recording_id` -- index of the recording segment to analyse.
    * Only one segment should be analysed at a time - do not specify a
      list or range!
    * Note that this index is relative to the data segment(s) chosen in
      the above step. (e.g. If `data_segments = [0, 2]` was used
      previously, then `recording_id = 1` should be used to inspect the
      third segment from the original file.)
* `epoch_timing_ms` -- timing window in milliseconds relative to each
    stimulus onset during which spikes may occur, as a `tuple` of
    numeric values.
    * The first value indicates start time and the second value end
      time.
    * The conversion from milliseconds to samples is done using
      `int()`,  which always rounds down non-integer floats. Since
      the expected sample rate is high, this level of imprecision
      should not be important. However, note that it may be possible
      for floating-point imprecision to cause misalignment of the
      epochs and spikes by one sample.
* `threshold_uV` -- threshold in microvolts above which spikes
  should be detected.

In [ ]:
# User defined variables
recording_id = 0
epoch_timing_ms = (1, 5)
threshold_uV = 0


### User does not need to modify below this line ###

try:
    recording = recordings[recording_id]
except (AttributeError, TypeError) as e:
    raise TypeError(
        "Run this block on one recording segment at a time (i.e. "
        "ensure that `recording_id` is an `int`, not a `range`)."
    ) from e
(spikes, epochs) = hff_analysis.spikes_info(
    recording,
    repetition,
    epoch_timing_ms,
    threshold_uV
)

# STEP 3: FILTER AND PLOT SPIKES

This code block filters detected spikes and plots them. Run this block,
then tweak `spike_criteria` using the plots to exclude detected peaks
which are not from the target unit. Repeat as many times as necessary
before moving onto step 4.

## Arguments
* `spike_criteria` -- a dictionary containing `SpikeCriteria` objects
  for each stimulation type.
* `exclude_frequencies` -- a list of frequencies to exclude.
* `exclude_amplitudes` -- a list of amplitudes to exclude.
* `save_figures` -- a bool for controlling whether figures are saved.
  If this is enabled, existing figures at the target path will be
  overwritten.

In [ ]:
# User defined variables
spike_criteria = {
    'mechanical': {
        'latency_min_ms': 3.2,
        'latency_max_ms': 4,
        'size_min_uV': 50,
        'size_max_uV': None
    },
    'electrical': {
        'latency_min_ms': 2.6,
        'latency_max_ms': 3.5,
        'size_min_uV': 70,
        'size_max_uV': None
    }
}
exclude_frequencies = []
exclude_amplitudes = []
save_figures = True


### User does not need to modify below this line ###

recording_segment = (data_segments[recording_id] if data_segments
                     else recording_id)
filtered_spikes = hff_analysis.filter_spikes(
    spikes,
    spike_criteria,
    exclude_frequencies,
    exclude_amplitudes
)
hff_analysis.plot_clusters(
    filtered_spikes,
    epochs,
    repetition,
    recording_segment,
    exclude_frequencies,
    exclude_amplitudes,
    save_figures
)

# STEP 4: SAVE DATA
Run this code block once after you are satisfied with the spike
detection to save your results as a JSON file. Repeat these 3 steps for
each file/recording segment that will be analysed, then move onto
`main_part2.ipynb`.

By default, if the target file already exists, the user will be asked
for manual confirmation before the file is overwritten. This behaviour
can be changed by setting `force_overwrite = True` at the start
of the code block.

In [ ]:
# User defined variables
force_overwrite = True


### User does not need to modify below this line ###

hff_analysis.save_to_json(
    filename,
    repetition,
    recording_segment,
    'frs',
    force_overwrite,
    filename=filename,
    epoch_timing_ms=epoch_timing_ms,
    threshold_uV=threshold_uV,
    spike_criteria=spike_criteria,
    exclude_frequencies=exclude_frequencies,
    exclude_amplitudes=exclude_amplitudes
)
hff_analysis.save_to_json(
    filename,
    repetition,
    recording_segment,
    'epochs',
    force_overwrite,
    epochs=epochs,
    exclude_frequencies=exclude_frequencies,
    exclude_amplitudes=exclude_amplitudes
)
hff_analysis.save_to_json(
    filename,
    repetition,
    recording_segment,
    'spikes',
    force_overwrite,
    spikes=filtered_spikes
)